In [ ]:
import sys
sys.path.append('../..')

from data.utils import get_datasets
import torch
from models.cnn import CNN
import importlib
importlib.reload(sys.modules['models.train'])
from models.train import train, predict
import importlib
import models.evaluate
importlib.reload(models.evaluate)
from models.evaluate import evaluate
from plots import plot_training_curves, plot_confusion_matrix, plot_per_class_metrics
importlib.reload(sys.modules['models.evaluate'])

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print("Using device:", device)

In [ ]:
train_ds, valid_ds, test_ds = get_datasets('mfcc', train_path='../../data/train', valid_path='../../data/valid', test_path='../../data/test')

In [ ]:
cnn = CNN(num_classes=12).to(device)
total_params = sum(p.numel() for p in cnn.parameters())
print(f'Parameters: {total_params:,}')

In [ ]:
model, history = train(cnn, train_ds, valid_ds, epochs=30, batch_size=64, lr=3e-4, device=device, checkpoint_name='cnn_mfcc')

In [ ]:
plot_training_curves(history, title='CNN – MFCC')

In [ ]:
preds_valid, labels_valid = predict(model, valid_ds, device=device)
result_valid = evaluate(preds_valid, labels_valid)
plot_confusion_matrix(result_valid['cm'], normalize=True, title='CNN – MFCC (valid)')
plot_per_class_metrics(labels_valid.numpy(), preds_valid.numpy(), title='CNN – MFCC (valid)')

In [ ]:
preds_test, labels_test = predict(model, test_ds, device=device)
result_test = evaluate(preds_test, labels_test)
plot_confusion_matrix(result_test['cm'], normalize=True, title='CNN – MFCC (test)')
plot_per_class_metrics(labels_test.numpy(), preds_test.numpy(), title='CNN – MFCC (test)')